[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C56_Detection_Augmentation_Course/00_setup/00_environment_check.ipynb)

# 00 · 检测增强的地基（不变性先验 / 三判据 / 标注同步 / 正确性断言）

本课全程 **纯 numpy + 标准库、CPU、不联网**。不需要 OpenCV / albumentations / PyTorch。

**核心命题**：数据增强不是「造更多数据」，而是**往模型里注入一条「这些变化不改变标签」的先验**。
既然是先验，它就**可能是错的**——而错误的先验比缺少数据更有害，因为模型会稳定地、自信地学会它。

这个 notebook 你会亲手实现：
1. **bbox 的四种表示与互转**（xyxy / cxcywh / xywh / 归一化），以及 IoU
2. **检测增强的通用框架**：每个算子必须同时给出 `apply_image` 与 `apply_boxes`
3. **「忘记变换标注」的量化后果**——图看着完全正常，框的 IoU 已经是 0
4. **三判据自检清单**：保标签语义 / 贴近部署分布 / 不制造训练-推理鸿沟
5. **可执行的正确性断言**：用目标掩码算出「真实紧框」，与算子输出对拍

> 心智模型：**图像和标注是两个必须同步变换的对象。本课所有的坑都从这一句话长出来。**

## 1 · 环境自检

In [ ]:
import sys, platform, math, json, itertools, collections
print('Python', sys.version.split()[0], '|', platform.system(), platform.machine())
import numpy as np
print('numpy', np.__version__)
for name in ['cv2', 'albumentations', 'torch']:
    try:
        m = __import__(name)
        print(f'  {name:<16s} {getattr(m, "__version__", "?")}  (有更好，但本课不用)')
    except ImportError:
        print(f'  {name:<16s} 未安装 -> 走 numpy 手写路径（本课的默认路径）')

rng = np.random.default_rng(0)
print('\n环境就绪 ✅  —— 本课不需要 OpenCV / albumentations / GPU / 联网')

## 2 · bbox 的四种表示与互转

真实项目里 bbox 至少有四种写法，它们互相之间的转换是**数据管线 bug 的第一大来源**
（C61 模块 03 会把「坐标格式弄反」列为「mAP 恒为 0」的三大成因之一）。

| 表示 | 含义 | 谁在用 |
|---|---|---|
| `xyxy` | `(x1, y1, x2, y2)` 左上/右下 | torchvision、mmdet 内部、大多数评测代码 |
| `xywh` | `(x1, y1, w, h)` 左上 + 宽高 | **COCO 标注文件** |
| `cxcywh` | `(cx, cy, w, h)` 中心 + 宽高 | DETR 家族、YOLO 的回归目标 |
| `cxcywh_norm` | 上面除以 `(W, H)` 归一化到 `[0,1]` | **YOLO 的 `.txt` 标注文件** |

约定：本课全程用 `xyxy`，且 **`x2/y2` 是开区间**（`x2 - x1` 就是宽度，不需要 `+1`）。
这个约定必须写在文档里——「要不要 +1」在 NMS 与 IoU 实现里会造成微妙差异（C60 模块 04）。

In [ ]:
def xyxy2cxcywh(b):
    b = np.asarray(b, float).reshape(-1, 4)
    return np.stack([(b[:, 0] + b[:, 2]) / 2, (b[:, 1] + b[:, 3]) / 2,
                     b[:, 2] - b[:, 0],       b[:, 3] - b[:, 1]], axis=1)

def cxcywh2xyxy(b):
    b = np.asarray(b, float).reshape(-1, 4)
    return np.stack([b[:, 0] - b[:, 2] / 2, b[:, 1] - b[:, 3] / 2,
                     b[:, 0] + b[:, 2] / 2, b[:, 1] + b[:, 3] / 2], axis=1)

def xyxy2xywh(b):
    b = np.asarray(b, float).reshape(-1, 4)
    return np.stack([b[:, 0], b[:, 1], b[:, 2] - b[:, 0], b[:, 3] - b[:, 1]], axis=1)

def normalize(b, W, H):
    b = np.asarray(b, float).reshape(-1, 4).copy()
    b[:, [0, 2]] /= W; b[:, [1, 3]] /= H
    return b

def box_area(b):
    b = np.asarray(b, float).reshape(-1, 4)
    return np.clip(b[:, 2] - b[:, 0], 0, None) * np.clip(b[:, 3] - b[:, 1], 0, None)

def box_iou(a, b):
    '''a:(N,4) b:(M,4) -> (N,M)'''
    a = np.asarray(a, float).reshape(-1, 4); b = np.asarray(b, float).reshape(-1, 4)
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    union = box_area(a)[:, None] + box_area(b)[None, :] - inter
    return inter / np.maximum(union, 1e-12)

boxes = np.array([[13., 11., 28., 26.], [42., 36., 51., 45.]])
print('xyxy       ', boxes.tolist())
print('cxcywh     ', xyxy2cxcywh(boxes).tolist())
print('xywh (COCO)', xyxy2xywh(boxes).tolist())
print('归一化 cxcywh (YOLO txt)', np.round(normalize(xyxy2cxcywh(boxes), 64, 64), 4).tolist())
print('面积       ', box_area(boxes).tolist())

assert np.allclose(cxcywh2xyxy(xyxy2cxcywh(boxes)), boxes), '往返必须无损'
assert np.allclose(np.diag(box_iou(boxes, boxes)), 1.0)
assert box_iou(boxes[:1], boxes[1:])[0, 0] == 0.0, '两个不相交的框 IoU=0'
print('\n✅ 四种表示互转与 IoU 就位（全程 xyxy，x2/y2 为开区间）')

## 3 · 检测增强的通用框架：两条轨道必须同步

分类增强的标签变换 `g_T` 是**恒等映射**，不可能写错。
检测增强的 `g_T` 是一个**真正的函数**——而它写错的时候，**图看起来完全正常**。

下面先造一个可验证的合成场景：一张 64×64 的灰度图，上面有两个**圆形交通标志**
（中国的禁令标志就是圆形）。因为我们知道每个标志的解析形状，
就可以随时从图像里算出「**真实的紧框**」，用它来给任何算子的 `apply_boxes` 打分。

In [ ]:
def make_scene(H=64, W=64, signs=((20, 18, 7, 1), (46, 40, 4, 2))):
    '''signs: (cx, cy, r, cls)，每个是一个圆形标志。
       返回 img(uint8 灰度), boxes(xyxy, x2/y2 开区间), labels, 以及每类的像素值。'''
    img = np.zeros((H, W), np.uint8)
    yy, xx = np.mgrid[0:H, 0:W]
    boxes, labels, vals = [], [], []
    for cx, cy, r, cls in signs:
        val = 100 + 40 * cls                     # 每个标志一个唯一像素值，方便反查掩码
        img[(xx - cx) ** 2 + (yy - cy) ** 2 <= r * r] = val
        boxes.append([cx - r, cy - r, cx + r + 1, cy + r + 1])   # 圆占 [cx-r, cx+r] 共 2r+1 列
        labels.append(cls); vals.append(val)
    return img, np.array(boxes, float), np.array(labels, int), vals

def mask_bbox(img, val):
    '''从图像里反查某个目标的**真实紧框** —— 这是我们的 ground truth 检查器。'''
    ys, xs = np.nonzero(img == val)
    assert len(xs) > 0, f'像素值 {val} 在图里找不到'
    return np.array([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1], float)

img, boxes, labels, vals = make_scene()
print('图像', img.shape, '| 目标数', len(boxes), '| 类别', labels.tolist())
for b, v in zip(boxes, vals):
    tb = mask_bbox(img, v)
    print(f'  标注框 {b.tolist()}   掩码紧框 {tb.tolist()}   IoU={box_iou([b],[tb])[0,0]:.4f}')

assert all(np.allclose(mask_bbox(img, v), b) for b, v in zip(boxes, vals)), '初始标注必须与掩码完全一致'
print('\n✅ 合成场景就位：标注与掩码逐像素一致，可以用来给任何增强算子打分。')

In [ ]:
# ── 通用框架：每个算子必须同时给出 apply_image 与 apply_boxes ──
class Aug:
    '''检测增强算子的最小接口约定。**两条轨道，同一个 T。**'''
    name = 'identity'
    def apply_image(self, img):            return img
    def apply_boxes(self, boxes, shape):   return boxes
    def apply_labels(self, labels):        return labels          # 少数算子需要改类别（模块 01）
    def __call__(self, img, boxes, labels):
        H, W = img.shape[:2]
        return (self.apply_image(img),
                self.apply_boxes(boxes, (H, W)),
                self.apply_labels(labels))

class HFlip(Aug):
    name = 'hflip'
    def apply_image(self, img):  return img[:, ::-1].copy()
    def apply_boxes(self, boxes, shape):
        H, W = shape
        b = np.asarray(boxes, float).copy()
        b[:, [0, 2]] = np.stack([W - boxes[:, 2], W - boxes[:, 0]], axis=1)   # 注意 x1/x2 会互换
        return b

class Translate(Aug):
    name = 'translate'
    def __init__(self, dx, dy): self.dx, self.dy = int(dx), int(dy)
    def apply_image(self, img):
        H, W = img.shape; dx, dy = self.dx, self.dy
        out = np.zeros_like(img)
        xs0, xs1 = max(0, dx), min(W, W + dx)
        ys0, ys1 = max(0, dy), min(H, H + dy)
        out[ys0:ys1, xs0:xs1] = img[ys0 - dy:ys1 - dy, xs0 - dx:xs1 - dx]
        return out
    def apply_boxes(self, boxes, shape):
        b = np.asarray(boxes, float).copy()
        b[:, [0, 2]] += self.dx; b[:, [1, 3]] += self.dy
        return b

def verify(op, img, boxes, vals, tol=0.99):
    '''**可执行的正确性断言**：算子输出的框 vs 掩码算出的真实紧框。'''
    img2, boxes2, _ = op(img, boxes, np.zeros(len(boxes), int))
    ious = [box_iou([b], [mask_bbox(img2, v)])[0, 0] for b, v in zip(boxes2, vals)]
    return img2, boxes2, np.array(ious), all(i >= tol for i in ious)

for op in [HFlip(), Translate(5, -3)]:
    _, b2, ious, ok = verify(op, img, boxes, vals)
    print(f'{op.name:<10s} 变换后框 {np.round(b2,1).tolist()}  与掩码紧框 IoU={np.round(ious,4).tolist()}  {"✅" if ok else "❌"}')
    assert ok, op.name
print('\n✅ 两条轨道同步：断言用「掩码算出的真实紧框」验证，而不是靠肉眼看图。')

In [ ]:
# ── 反面教材：图变了、标注没变。**loss 照降，图看着完全正常。** ──
class BuggyHFlip(HFlip):
    name = 'hflip(忘记变框)'
    def apply_boxes(self, boxes, shape): return np.asarray(boxes, float).copy()

class BuggyTranslate(Translate):
    name = 'translate(忘记变框)'
    def apply_boxes(self, boxes, shape): return np.asarray(boxes, float).copy()

print(f"{'算子':<24s} {'与真实紧框的 IoU':>22s}   后果")
rows = []
for op, why in [(BuggyHFlip(), '整个目标跑到了对侧'),
                (BuggyTranslate(5, -3), '只偏了 5 像素，图几乎看不出区别')]:
    _, _, ious, ok = verify(op, img, boxes, vals)
    rows.append((op.name, ious))
    print(f'{op.name:<24s} {str(np.round(ious,3).tolist()):>22s}   {why}')

iou_flip = rows[0][1]; iou_tr = rows[1][1]
assert iou_flip.max() == 0.0, '翻转后不改框 -> 框与目标完全不重叠'
assert 0.3 < iou_tr[0] < 0.45, f'15px 的框偏 5px，IoU 应在 0.36 附近，实得 {iou_tr[0]:.3f}'
print(f'\n⚠️  第二行是真正危险的那个：图肉眼看不出区别，但')
print(f'    15×15 的框偏 5 px -> IoU {iou_tr[0]:.2f}；**9×9 的框偏同样的 5 px -> IoU {iou_tr[1]:.2f}**。')
print('    大目标偏 5 px 还能勉强算对，**小目标偏 5 px 标签就基本作废** —— 这是 TSR 的常态：')
print('    60 米外的限速牌在 1080p 上只有十几个像素（C57 模块 05 会做完整的物理推导）。')
print('✅ 结论：检测增强的 bug 不会崩溃、不会 NaN、不会让 loss 异常，')
print('   它只让指标低一点点 —— 低到你以为是「模型不够好」。**所以必须写断言。**')

## 4 · 三判据自检清单

一个增强该不该用，按顺序检查三条，**第一条不过后两条就不用看了**：

1. **保标签语义** —— 变换后原标签还是正确答案吗？（不过 = 正确性问题，模型会稳定学错）
2. **贴近部署分布** —— 车上那台相机有可能拍到这种图吗？（不过 = 效率问题，浪费容量）
3. **不制造训练-推理鸿沟** —— 推理端会做同样的事吗？没有的话差异对齐了吗？（不过 = 交付问题）

In [ ]:
class AugSpec:
    def __init__(self, name, label_safe, deploy_realistic, gap_free, note=''):
        self.name = name
        self.label_safe = label_safe            # 判据①
        self.deploy_realistic = deploy_realistic  # 判据②
        self.gap_free = gap_free                # 判据③
        self.note = note

def audit(spec):
    '''返回 (verdict, mitigation)。**按危害量级排序：① > ② > ③。**'''
    if not spec.label_safe:
        return '❌ 禁用/改造', '必须先修标签变换：类别白名单、标签互换、或改用 ignore 区域'
    if not spec.deploy_realistic:
        return '⚠️ 收益存疑', '部署分布里不存在这种图；要么砍掉，要么把幅度收到真实范围内'
    if not spec.gap_free:
        return '⚠️ 需显式对齐', '训练与推理必须用同一实现（或训练末期关闭），并写进部署对拍清单'
    return '✅ 直接用', '—'

SPECS = [
    AugSpec('水平翻转（通用 COCO 场景）',   True,  True,  True,  '左右镜像不改变「猫」的类别'),
    AugSpec('水平翻转（TSR，无白名单）',    False, True,  True,  '「向左转弯」被镜像成「向右转弯」，标签却没变'),
    AugSpec('垂直翻转',                    True,  False, True,  '车载相机不会上下颠倒；重力是极强的先验'),
    AugSpec('随机缩放 0.5–1.5×',           True,  True,  True,  '对应标志的远近 —— TSR 收益最高的几何增强'),
    AugSpec('旋转 ±10°',                   True,  True,  True,  '对应车身侧倾与相机安装误差；但外接框会膨胀'),
    AugSpec('旋转 ±45°',                   True,  False, True,  '真实驾驶中不存在；且正方形框面积膨胀 100%'),
    AugSpec('letterbox 缩放+填充',         True,  True,  False, '训练与推理端实现必须逐像素一致'),
    AugSpec('HSV 色相抖动 ±30°',           False, True,  True,  '红色禁令牌被抖成蓝色 —— 颜色即语义'),
    AugSpec('Mosaic 四图拼接',             True,  False, False, '拼接图在真实分布里不存在 -> 末期必须 close-mosaic'),
    AugSpec('Copy-Paste 稀有类',           True,  True,  True,  '需约束尺度与位置，否则贴出「天上的标志」'),
    AugSpec('测试时增强 TTA',              True,  True,  False, '训练端没有；车端多跑几次的延迟不可接受'),
]

print(f"{'增强算子':<26s} {'①语义':>6s} {'②分布':>6s} {'③无鸿沟':>7s}  {'结论':<12s} 说明")
n_ok = n_ban = 0
for s in SPECS:
    v, mit = audit(s)
    n_ok += (v.startswith('✅')); n_ban += (v.startswith('❌'))
    tick = lambda x: ' ✓' if x else ' ✗'
    print(f'{s.name:<26s} {tick(s.label_safe):>6s} {tick(s.deploy_realistic):>6s} '
          f'{tick(s.gap_free):>7s}  {v:<12s} {s.note}')

assert audit(SPECS[1])[0].startswith('❌'), 'TSR 无白名单翻转必须被禁'
assert audit(SPECS[7])[0].startswith('❌'), 'HSV 大幅色相抖动破坏「颜色即语义」'
assert n_ok == 4 and n_ban == 2, (n_ok, n_ban)
print(f'\n11 个算子里：{n_ok} 个可直接用，{n_ban} 个必须先改造，其余 {11-n_ok-n_ban} 个需要缓解措施。')
print('⚠️  注意「水平翻转」在通用场景 ✅、在 TSR ❌ —— **同一个算子，判据结论相反。**')
print('    这就是为什么「照抄一份 COCO 的增强配置」在 TSR 上是错的。')

In [ ]:
# ── 缓解措施清单：非 ✅ 的算子分别要做什么 ──
print('需要处理的算子与对应动作：\n')
for s in SPECS:
    v, mit = audit(s)
    if not v.startswith('✅'):
        print(f'  {v}  {s.name}')
        print(f'        -> {mit}')
        print()

mitigations = {s.name: audit(s)[1] for s in SPECS if not audit(s)[0].startswith('✅')}
assert len(mitigations) == 7
assert '白名单' in mitigations['水平翻转（TSR，无白名单）']
assert '部署分布' in mitigations['Mosaic 四图拼接'], 'Mosaic 先卡在判据②（拼接图不是真实分布）'
print('✅ 三判据清单的价值：把「这个增强好像不太对」变成「它违反了第 N 条，对应动作是 X」。')
print('   面试里被问「你怎么选增强」，这张表就是一个可以直接讲的框架。')

## 5 · 判据①的可执行形式：标签语义是否被保住

前面 `verify()` 检查的是**几何**同步（框还紧贴目标吗）。
但判据① 还有**语义**的一半：**类别标签是否还正确**。

几何检查通不过 → 代码 bug；语义检查通不过 → **先验错了**。后者严重得多，
因为它不会在任何自动检查里冒出来 —— 除非你像下面这样，把「类别的镜像映射」显式写出来。

In [ ]:
# TSR 的三档翻转分类（模块 01 会做成完整白名单并给出判定流程）
# 'safe'  : 外观左右镜像对称 + 语义无方向性        -> 可以翻
# 'swap'  : 语义含方向性，但标签集里存在镜像类     -> 可以翻，**但必须同时换标签**
# 'forbid': 其余                                   -> 绝对不能翻
FLIP3 = {
    'no_entry':       ('safe',   None),          # 禁止驶入：红底白横杠，左右对称
    'no_vehicles':    ('safe',   None),          # 禁止通行
    'turn_left':      ('swap',   'turn_right'),  # 向左转弯 <-> 向右转弯
    'turn_right':     ('swap',   'turn_left'),
    'speed_limit_60': ('forbid', '含数字，镜像后的字形在现实中不存在'),
    'stop':           ('forbid', '含文字 STOP/停'),
    'guide_sign':     ('forbid', '指路牌含地名文字'),
}

def flip_labels(labels):
    '''语义层面的 g_T。返回 (新标签, 是否合法)。'''
    out = []
    for c in labels:
        kind, other = FLIP3[c]
        if kind == 'forbid':
            return labels, False
        out.append(other if kind == 'swap' else c)
    return out, True

for scene in [['no_entry', 'no_vehicles'], ['turn_left', 'no_entry'],
              ['turn_left', 'speed_limit_60'], ['stop']]:
    new, ok = flip_labels(scene)
    print(f'{str(scene):<38s} -> {"可翻 " + str(new) if ok else "❌ 整张图不能翻（含 forbid 类）"}')

assert flip_labels(['turn_left'])[0] == ['turn_right'], '方向类必须换标签'
assert not flip_labels(['turn_left', 'speed_limit_60'])[1], '一张图里有一个 forbid 就整张不能翻'
assert flip_labels(['no_entry', 'no_vehicles'])[1]
print('\n⚠️  注意第 3 行：**一张图里只要有一个目标属于 forbid，整张图就不能翻**。')
print('    因为翻转作用在图像上，而判定是逐目标的 —— 这个「与」的关系会让')
print('    TSR 里翻转的实际生效率远低于配置里写的 p_flip（模块 01 会量化）。')
print('✅ 判据① 的两半：几何同步（写断言）+ 语义映射（写白名单）。缺一不可。')

## ✏️ 练习 1：bbox 工具函数

实现 `boxes_from_yolo_txt(lines, W, H)`：把 YOLO 标注文件的每行
`"cls cx cy w h"`（**归一化的 cxcywh**）解析成 `(labels, boxes_xyxy)`，
其中 `boxes_xyxy` 是**像素坐标**的 `xyxy`。

再实现 `filter_tiny(boxes, labels, min_side)`：丢掉宽或高小于 `min_side` 像素的框，
返回 `(boxes_kept, labels_kept, n_dropped)`。

In [ ]:
def boxes_from_yolo_txt(lines, W, H):
    # TODO: 每行 "cls cx cy w h"，cx/cy/w/h 已按 W,H 归一化
    #       -> 返回 (labels: np.int64 (N,), boxes: np.float64 (N,4) 像素 xyxy)
    #       空输入要返回 (shape (0,) 的 int 数组, shape (0,4) 的 float 数组)
    raise NotImplementedError

def filter_tiny(boxes, labels, min_side):
    # TODO: 保留 (w >= min_side) & (h >= min_side) 的框
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
LINES = ['0 0.5 0.5 0.25 0.25',      # 640x640 -> [240,240,400,400]
         '3 0.1 0.2 0.01 0.01',      # -> 6.4x6.4 像素的小标志
         '7 0.9 0.8 0.05 0.05']      # -> 32x32
lab, bx = boxes_from_yolo_txt(LINES, 640, 640)
assert lab.tolist() == [0, 3, 7], lab
assert np.allclose(bx[0], [240., 240., 400., 400.]), bx[0]
assert np.allclose(box_area(bx[1:2])[0], 6.4 * 6.4), box_area(bx[1:2])
l0, b0 = boxes_from_yolo_txt([], 640, 640)
assert b0.shape == (0, 4) and l0.shape == (0,), (b0.shape, l0.shape)

kb, kl, nd = filter_tiny(bx, lab, min_side=8.0)
assert nd == 1 and kl.tolist() == [0, 7], (nd, kl)
kb2, kl2, nd2 = filter_tiny(bx, lab, min_side=2.0)
assert nd2 == 0
print('原始 3 个目标，尺寸(w):', np.round(bx[:, 2] - bx[:, 0], 2).tolist())
print('min_side=8 -> 丢弃', nd, '个；min_side=2 -> 丢弃', nd2, '个')
print('⚠️  同一份标注，只因为 min_side 从 2 改成 8，就少了 1/3 的目标 ——')
print('    而被丢掉的恰恰是**最难、最有价值的远处小标志**。（模块 01 会深挖这一点）')
print('✅ 练习 1 通过')

## ✏️ 练习 2：三判据审计器

实现 `audit2(label_safe, deploy_realistic, gap_free)`，返回
`(level, action)`，其中 `level ∈ {'ban', 'mitigate', 'ok'}`：

- 判据① 不过 → `('ban', '修标签变换')`
- 判据① 过、②或③ 不过 → `('mitigate', ...)`：②不过给 `'收窄幅度'`，③不过给 `'两端对齐'`；
  **两个都不过时优先报 `'收窄幅度'`**（危害更大）
- 三条都过 → `('ok', '直接用')`

In [ ]:
def audit2(label_safe, deploy_realistic, gap_free):
    # TODO
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert audit2(False, True,  True)  == ('ban', '修标签变换')
assert audit2(False, False, False) == ('ban', '修标签变换'), '判据①优先级最高'
assert audit2(True,  False, True)  == ('mitigate', '收窄幅度')
assert audit2(True,  True,  False) == ('mitigate', '两端对齐')
assert audit2(True,  False, False) == ('mitigate', '收窄幅度'), '②③都不过时报②'
assert audit2(True,  True,  True)  == ('ok', '直接用')

print(f"{'算子':<26s} {'level':<10s} action")
for s in SPECS:
    lv, ac = audit2(s.label_safe, s.deploy_realistic, s.gap_free)
    print(f'{s.name:<26s} {lv:<10s} {ac}')
levels = [audit2(s.label_safe, s.deploy_realistic, s.gap_free)[0] for s in SPECS]
assert collections.Counter(levels) == {'ok': 4, 'mitigate': 5, 'ban': 2}, collections.Counter(levels)
print('\n✅ 练习 2 通过：4 个直接可用 / 5 个需缓解 / 2 个必须先改造')

## ✏️ 练习 3：写一个算子并让断言通过

实现 `class Scale(Aug)`：把图像**整数倍缩小** `k` 倍（用最近邻：`img[::k, ::k]`），
并给出正确的 `apply_boxes`。

提示：`img[::k, ::k]` 取的是原图第 `0, k, 2k, …` 行/列，
所以原坐标 `x` 映射到新坐标 `x / k`；框的 `x1,y1,x2,y2` 全部除以 `k` 即可。
断言用 `verify()`，容差放到 `0.70`（最近邻降采样必然引入亚像素误差，
**这个容差本身就是一条重要信息：降采样会让小目标的标注精度下降**）。

In [ ]:
class Scale(Aug):
    name = 'scale'
    def __init__(self, k): self.k = int(k)
    def apply_image(self, img):
        # TODO
        raise NotImplementedError
    def apply_boxes(self, boxes, shape):
        # TODO
        raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
op = Scale(2)
img2, b2, ious, ok = verify(op, img, boxes, vals, tol=0.70)
print('缩小 2 倍后图像尺寸', img2.shape, '| 框', np.round(b2, 2).tolist())
print('与掩码紧框的 IoU     ', np.round(ious, 4).tolist())
assert img2.shape == (32, 32), img2.shape
assert np.allclose(b2, boxes / 2), '框应整体除以 k'
assert ok, f'IoU 应 >= 0.70，实得 {ious}'
assert ious.min() < 0.999, '最近邻降采样必然带来亚像素误差 —— 这正是要点'

big, small = ious[0], ious[1]
print(f'\n大标志(15px->7.5px) IoU={big:.3f} | 小标志(9px->4.5px) IoU={small:.3f}')
assert small <= big + 1e-9, '越小的目标，降采样带来的相对标注误差越大'
print('⚠️  同一次降采样，**小目标的标注精度损失更大** ——')
print('    这是 C57「标注误差的相对量级」那条的一个直接演示：')
print('    人工标注 ±1px 的误差，对 8px 的框就是 12.5% 的相对误差，标签本身就带噪。')
print('✅ 练习 3 通过')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def boxes_from_yolo_txt(lines, W, H):
    if not lines:
        return np.zeros(0, np.int64), np.zeros((0, 4), float)
    labs, cxywh = [], []
    for ln in lines:
        parts = ln.split()
        labs.append(int(parts[0]))
        cx, cy, w, h = (float(v) for v in parts[1:5])
        cxywh.append([cx * W, cy * H, w * W, h * H])
    return np.array(labs, np.int64), cxcywh2xyxy(np.array(cxywh, float))

def filter_tiny(boxes, labels, min_side):
    boxes = np.asarray(boxes, float).reshape(-1, 4)
    w = boxes[:, 2] - boxes[:, 0]; h = boxes[:, 3] - boxes[:, 1]
    keep = (w >= min_side) & (h >= min_side)
    return boxes[keep], np.asarray(labels)[keep], int((~keep).sum())

In [ ]:
# 练习 2 参考答案
def audit2(label_safe, deploy_realistic, gap_free):
    if not label_safe:
        return 'ban', '修标签变换'
    if not deploy_realistic:
        return 'mitigate', '收窄幅度'
    if not gap_free:
        return 'mitigate', '两端对齐'
    return 'ok', '直接用'

In [ ]:
# 练习 3 参考答案
class Scale(Aug):
    name = 'scale'
    def __init__(self, k): self.k = int(k)
    def apply_image(self, img):
        return img[::self.k, ::self.k].copy()
    def apply_boxes(self, boxes, shape):
        return np.asarray(boxes, float) / self.k

---
## 🧪 真实工程胶囊：检测增强算子的接口约定与审计表

下面这段可以原样复制进你自己的项目（把 numpy 换成 OpenCV / albumentations 即可）。
它的价值不在代码，在于**把「这个增强对标注做了什么」变成必填字段**。

In [ ]:
RECIPE = r'''
# ============================================================
# 检测增强算子的接口约定（放进 project/augment/base.py）
# ============================================================
class DetAug:
    # 任何检测增强算子都必须实现下面的方法，且图像与标注**必须成对**变换。
    # 铁律：
    #   1. 改了图像几何       -> 必须实现 apply_boxes
    #   2. 改了图像语义(镜像/颜色) -> 必须实现 apply_labels 或声明类别白名单
    #   3. 可能让目标越界     -> 必须声明 OOB_POLICY in {keep, ignore, drop}

    # ---- 三判据（提交 PR 时必须填，reviewer 按此审）----
    LABEL_SAFE       = None       # 判据①：变换后原标签仍正确？(False 则必须写白名单/标签映射)
    DEPLOY_REALISTIC = None       # 判据②：车载相机可能拍到这种图？
    GAP_FREE         = None       # 判据③：推理端也做同样的事？否则怎么对齐？
    OOB_POLICY       = 'ignore'   # 越界目标：keep / ignore / drop
    NOTE             = ''         # 对应部署分布里的什么现象

    def apply_image(self, img):           raise NotImplementedError
    def apply_boxes(self, boxes, shape):  return boxes
    def apply_labels(self, labels):       return labels

# ============================================================
# 单元测试模板：**每个几何算子都必须有这个测试**
# 用目标掩码算出的「真实紧框」验证 apply_boxes —— 不要靠可视化抽查。
# ============================================================
def test_geometry_sync(op, img, boxes, masks, tol=0.95):
    img2   = op.apply_image(img)
    boxes2 = op.apply_boxes(boxes, img.shape[:2])
    for b, m in zip(boxes2, masks):
        m2 = op.apply_image(m.astype(img.dtype))
        ys, xs = m2.nonzero()
        if len(xs) == 0:            # 目标被完全变换出画面：按 OOB_POLICY 处理，不算失败
            continue
        tight = [xs.min(), ys.min(), xs.max() + 1, ys.max() + 1]
        assert iou(b, tight) >= tol, 'apply_boxes 与 apply_image 不同步: ' + op.__class__.__name__

# ============================================================
# 增强配置的审计表（放进 configs/augment_audit.md，随配置一起 review）
# ============================================================
# | 算子 | ①语义 | ②分布 | ③无鸿沟 | 越界策略 | 缓解措施 | 对应的部署现象 |
# |------|-------|--------|---------|----------|----------|----------------|
# | RandomScale(0.5,1.5) | Y | Y | Y | ignore | -            | 标志的远近      |
# | RandomRotate(+-10)   | Y | Y | Y | ignore | 框膨胀已量化 | 车身侧倾/装配误差 |
# | HFlip                | N | Y | Y | -      | **类别白名单+标签互换** | (TSR 中几乎无效) |
# | Letterbox(640,114)   | Y | Y | N | keep   | **与 C++ 端逐像素对拍** | 输入尺寸归一 |
# | Mosaic               | Y | N | N | drop   | **close_mosaic=10 epoch** | (无对应现象) |
'''
print(RECIPE)
for k in ['LABEL_SAFE', 'DEPLOY_REALISTIC', 'GAP_FREE', 'OOB_POLICY',
          'test_geometry_sync', 'close_mosaic', '类别白名单']:
    assert k in RECIPE, k
print('✅ 胶囊覆盖：接口约定 / 三判据必填字段 / 越界策略 / 几何同步单元测试 / 配置审计表')

### 小结

- **增强 = 往模型里注入「这些变化不改变标签」的先验**，不是「造更多数据」。
  既然是先验就**可能是错的**，而错误的先验比缺少数据更有害——模型会稳定、自信地学会它。
- **三个判据，按危害排序**：① 保标签语义（正确性）→ ② 贴近部署分布（效率）→
  ③ 不制造训练-推理鸿沟（交付）。**第一条不过，后两条不用看。**
- **增强 / 正则化 / 更多数据不能互相替代**：增强注入的是任务先验，正则化注入的是复杂度偏好，
  只有真实数据带来新信息。**增强只能在已有样本的邻域里生成——没见过夜间就变不出真正的夜间。**
- **检测增强难在标签变换 `g_T` 不是恒等**：几何要同步、越界要判 keep/ignore/drop、
  尺度会改变小目标构成、类别可能需要跟着改。**而它写错的时候图看起来完全正常。**
- **所以每个算子都要有可执行的正确性断言**（掩码紧框 vs 输出框的 IoU），
  而不是靠可视化抽查——系统性偏移肉眼看不出来。
- **同一个算子在不同任务下判据结论可以相反**：水平翻转在 COCO 上 ✅、在 TSR 上 ❌。
  **照抄一份 COCO 增强配置就是这门课要防的第一个错误。**
- 一个 15×15 的框偏 5 像素，IoU 就掉到 0.36；**小目标对标注误差极其敏感**，
  而 TSR 的目标常年在十几个像素量级。

下一站：**模块 01 · 几何增强与标注同步** —— 仿射矩阵、四角法外接框、
**旋转导致的框膨胀**、越界三分法、**TSR 翻转白名单**、letterbox 正逆变换。